# SVD vs Normal Equation: Which is more robust?

We want to solve for $\mathbf{x}$ in:   

$\left[\begin{smallmatrix} & & \\ & \mathbf{A} & \\ & & \end{smallmatrix}\right]$  $\left[\begin{smallmatrix} \\ \mathbf{x} \\ \\ \end{smallmatrix}\right]$ = $\left[\begin{smallmatrix} \\ \mathbf{b} \\ \\ \end{smallmatrix}\right]$

where $\mathbf{A}$ is our design matrix and $\mathbf{b}$ is the target vector.

The condition number is a measures of how sensitive the solution of a linear system.
When $\mathbf{A}'\mathbf{A}$ has a high condition number, inverting it gives bad results.
SVD handles it gracefully.

In [8]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Create a random dataset with n samples and n features
n = 30
A = np.random.randn(n, n)  # Design matrix (n x n)
b = np.linspace(0, 1, len(A)) # Target vector (n,)

# Case 1: Well-conditioned X
print("CASE 1: Well-conditioned X")
print(f"  Condition number of X'X: {np.linalg.cond(A.T @ A):.2f}")


CASE 1: Well-conditioned X
  Condition number of X'X: 7208.51


The easiset and most straightforward apporach it to calculate $\mathbf{A}'$, to that the solution is found just by computing:

$\mathbf{x} =\mathbf{A}' \mathbf{b} $

This works only if $\mathbf{A}$ is a square matrix. If this is not true, the workaround is easily obtained by multiplying both sides by $\mathbf{A}^T$:

$\mathbf{x} = (\mathbf{A}^T \mathbf{A})' \mathbf{A}^T \mathbf{b}$

In [ ]:
x_normal = np.linalg.inv(A.T @ A) @ A.T @ b        # Normal Equation
print(f"RMSE Normal Eq.: {np.sqrt(np.mean((b - A @ x_normal)**2)):.6f}")

RMSE Normal Eq.: 0.000000


Using SVD instead, we first decompose $\mathbf{A}$ into:

$\mathbf{A} = \mathbf{U} \mathbf{S} \mathbf{V}^T$

and then calculate the solution as:

$\mathbf{x} = \mathbf{V} \mathbf{S}' \mathbf{U}^T \mathbf{b}$

In [ ]:
U, S, Vt = np.linalg.svd(A, full_matrices=False)
beta_svd = Vt.T @ np.linalg.inv(np.diag(S)) @ U.T @ b

print(f"RMSE SVD:        {np.sqrt(np.mean((b - A @ beta_svd)**2)):.6f}")
print(f"Are Normal Eq. and SVD equivalent? {np.allclose(x_normal, beta_svd)}")

RMSE SVD:        0.000000
Are Normal Eq. and SVD equivalent? True


In [53]:
# Case 2: Make two columns nearly identical (ill-conditioned) 
A[:, 0] = A[:, 1] + 1e-9* np.random.randn(len(A))

print(f"  Condition number of X'X: {np.linalg.cond(A.T @ A):.2e}")

x_normal = np.linalg.inv(A.T @ A) @ A.T @ b
U, S, Vt = np.linalg.svd(A, full_matrices=False)
x_svd_svd = Vt.T @ np.linalg.inv(np.diag(S)) @ U.T @ b

print(f"Are Normal Eq. and SVD equivalent? {np.allclose(x_normal, x_svd_svd)}")
print(f"RMSE Normal Eq.: {np.sqrt(np.mean((b - A @ x_normal)**2)):.6f}")
print(f"RMSE SVD:        {np.sqrt(np.mean((b - A @ x_svd_svd)**2)):.6f}")

  Condition number of X'X: 1.65e+16
Are Normal Eq. and SVD equivalent? False
RMSE Normal Eq.: 0.069215
RMSE SVD:        0.000000
